In [5]:
import pandas as pd

file_path = 'corpus.parquet'

df = pd.read_parquet("corpus.parquet", engine="fastparquet")
print(df.columns)
print("Total tokens:", df["token_ids"].map(len).sum())

Index(['token_ids'], dtype='object')
Total tokens: 5342208


In [7]:
df["token_ids"] = df["token_ids"].apply(
    lambda list: [int(token_id) for token_id in list]
)

In [8]:
from itertools import chain

token_ids = list(chain.from_iterable(df["token_ids"]))
print(len(token_ids), token_ids[:20])

5342208 [31206, 7229, 38387, 304, 74, 257, 4411, 4498, 45714, 2629, 282, 805, 285, 12171, 387, 72, 11, 264, 668, 2743]


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset (Dataset):
    def __init__(self, token_ids, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # data sampling with a sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1:i + max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [12]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

def dataloader(token_ids, batch_size, max_length, stride, shuffle = True, drop_last = True, num_workers = 0):
    
    dataset = GPTDataset(token_ids, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader

In [13]:
vocab_size = tokenizer.n_vocab
output_dim = 768

torch.manual_seed(123)

# Initialize token embedding layer
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [14]:
batch_size = 8
max_length = 1024
dataloader = dataloader(
    token_ids,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length,
    shuffle=True,
    drop_last=True,
    num_workers=0
)

In [15]:
context_length = max_length
# Initialize position embedding layer
position_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [16]:
# creating input embeddings
for batch in dataloader:
    input_ids, target_ids = batch

    token_embeddings = token_embedding_layer(input_ids)
    position_embeddings = position_embedding_layer(torch.arange(context_length))

    input_embeddings = token_embeddings + position_embeddings

    break

In [17]:
print(input_embeddings.shape)

torch.Size([8, 1024, 768])
